In [1]:
# Install dependencies (run once)
!pip install ultralytics torch torchvision

In [2]:
import os
import torch
from ultralytics import YOLO

In [3]:
import os

# Try multiple possible paths
possible_paths = [
    "smartvision_dataset/detection/data.yaml",
    "../data/detection/data.yaml",
    "data/detection/data.yaml"
]

DATA_YAML = None

for path in possible_paths:
    if os.path.exists(path):
        DATA_YAML = path
        break

if DATA_YAML is None:
    raise FileNotFoundError("❌ data.yaml not found! Check your folder structure")

print("✅ Found data.yaml at:", DATA_YAML)

✅ Found data.yaml at: ../data/detection/data.yaml


In [4]:
# Device
device = '0' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

Using device: cpu


In [5]:
# Load Model
model = YOLO('yolov8s.pt')

In [8]:
# Train Model
PROJECT_DIR = "runs/train"
RUN_NAME = "yolov8s_train"

results = model.train(
    data=DATA_YAML,
    epochs=10,
    imgsz=416,
    batch=4,
    device=device,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    verbose=True
)

print('Training Done!')

Ultralytics 8.4.62  Python-3.11.9 torch-2.12.0+cpu CPU (11th Gen Intel Core i5-11320H @ 3.20GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/detection/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tr

In [9]:
# Check and export weights
from pathlib import Path
import shutil

if "results" in globals() and hasattr(results, "save_dir"):
    run_dir = Path(results.save_dir)
else:
    # Fallback: locate latest run directory if this cell is executed separately
    candidates = sorted(Path.cwd().glob("**/runs/detect/*"), key=lambda p: p.stat().st_mtime)
    run_dir = candidates[-1] if candidates else None

if run_dir is None:
    print("Model not saved! No training run directory found.")
else:
    weights_dir = run_dir / "weights"
    best_model = weights_dir / "best.pt"
    last_model = weights_dir / "last.pt"

    print("YOLO run dir:", run_dir)

    if best_model.exists() or last_model.exists():
        chosen = best_model if best_model.exists() else last_model
        print("Saved model:", chosen)

        # Export to a stable path used by Streamlit pages
        export_dir = Path("..") / "runs" / "detect" / "smartvision_yolo_fast" / "weights"
        export_dir.mkdir(parents=True, exist_ok=True)
        export_path = export_dir / "last.pt"
        shutil.copy2(chosen, export_path)
        print("Exported model to:", export_path.resolve())
    else:
        print("Model not saved!")
        print("Expected under:", weights_dir.resolve())

YOLO run dir: D:\Miniproject\SpeedCamera\runs\detect\runs\train\yolov8s_train
Saved model: D:\Miniproject\SpeedCamera\runs\detect\runs\train\yolov8s_train\weights\best.pt
Exported model to: D:\VATHSAN\Gowtham\SmartVision_AI_System\runs\detect\smartvision_yolo_fast\weights\last.pt
